# Notebook 02 — Deep EDA & Statistical Diagnostics

Chạy hoàn toàn **offline** từ snapshot của Notebook 01:
1. Kiểm toán **26 ngày khuyết thời tiết** đối chiếu với `total_qty == 0` và `raw_holidays_rs.parquet`.
2. Tính bảng chẩn đoán thống kê 8 nhóm ATC (`ADI / CV²`, **Syntetos–Boylan**, `ADF`, `KPSS`, `Ljung–Box`, `Mann–Kendall`, `Shapiro–Wilk`, `ACF(1)_W`, `ACF(52)_W`) và ánh xạ sang **bảng quyết định chọn mô hình** (đặc biệt giải thích vì sao `N05C` cần **`Croston-SBA`** + **`LightGBM Tweedie`**).
3. Xuất `eda_diagnostics_8atc.csv` và 4 biểu đồ EDA 300 DPI.

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf, adfuller, kpss

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

from pathlib import Path
import os

def resolve_art_dir() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "BTL_BI").is_dir():
            d = p / "BTL_BI" / "artifacts" / "forecasting"
            d.mkdir(parents=True, exist_ok=True)
            return d
        if p.name == "BTL_BI":
            d = p / "artifacts" / "forecasting"
            d.mkdir(parents=True, exist_ok=True)
            return d
    d = cwd / "artifacts" / "forecasting"
    d.mkdir(parents=True, exist_ok=True)
    return d

ART_DIR = resolve_art_dir()
print("Artifacts dir:", ART_DIR)

df_sales = pd.read_parquet(ART_DIR / "raw_sales_5631.parquet")
df_weather = pd.read_parquet(ART_DIR / "raw_weather_5631.parquet")
df_hol = pd.read_parquet(ART_DIR / "raw_holidays_rs.parquet")

df_sales["local_date"] = pd.to_datetime(df_sales["local_date"])
df_weather["local_date"] = pd.to_datetime(df_weather["local_date"])
df_hol["local_date"] = pd.to_datetime(df_hol["local_date"])
print("Loaded snapshots:", len(df_sales), len(df_weather), len(df_hol))

In [ ]:
# Audit 26 ngày tổng bán cả 8 nhóm = 0 vs 26 ngày thiếu trong bảng thời tiết
exp_days = pd.date_range("2014-01-02", "2019-10-08", freq="D")
daily_tot = df_sales.groupby("local_date")["qty"].sum()
zero_dates = sorted(daily_tot[daily_tot == 0].index)
miss_weather = sorted(set(exp_days) - set(df_weather["local_date"]))

assert zero_dates == miss_weather, "Mismatch between 0-sale dates and missing weather dates!"
hol_map = dict(zip(df_hol["local_date"], df_hol["holiday_name"]))
df_zero_audit = pd.DataFrame([
    {
        "local_date": d.date(),
        "weekday": d.day_name(),
        "total_qty": float(daily_tot.loc[d]),
        "serbia_official_holiday": hol_map.get(d, "(Observed closure / Orthodox calendar)"),
    }
    for d in zero_dates
])
print(f"Verified EXACT match: {len(zero_dates)} zero-sale days == {len(miss_weather)} missing weather days.")
display(df_zero_audit)

In [ ]:
# Chẩn đoán Thống kê 8 Nhóm ATC (Daily & Weekly W-SUN = 302 tuần) + Ánh xạ Chọn Mô hình
df_weekly = (
    df_sales.set_index("local_date")
    .groupby("atc_code")["qty"]
    .resample("W-SUN")
    .sum()
    .reset_index()
)

diag_rows = []
for atc, g_d in df_sales.groupby("atc_code"):
    y_d = g_d.sort_values("local_date")["qty"].to_numpy()
    g_w = df_weekly[df_weekly["atc_code"] == atc].sort_values("local_date")
    y_w = g_w["qty"].to_numpy()

    nz = y_d[y_d > 0]
    adi = len(y_d) / len(nz)
    cv2 = (np.std(nz, ddof=1) / np.mean(nz)) ** 2
    if adi < 1.32 and cv2 < 0.49:
        sb_class = "Smooth"
    elif adi < 1.32 and cv2 >= 0.49:
        sb_class = "Erratic"
    elif adi >= 1.32 and cv2 < 0.49:
        sb_class = "Intermittent"
    else:
        sb_class = "Lumpy"

    acf_vals = acf(y_w, nlags=52, fft=False)
    adf_p = float(adfuller(y_w, autolag="AIC")[1])
    kpss_p = float(kpss(y_w, regression="c", nlags="auto")[1])
    lb_p = float(acorr_ljungbox(y_w, lags=[12], return_df=True)["lb_pvalue"].iloc[0])
    mk_tau, mk_p = stats.kendalltau(np.arange(len(y_w)), y_w)
    sw_p = float(stats.shapiro(y_w)[1])

    if sb_class in ("Intermittent", "Lumpy"):
        rec = "Croston-SBA + LightGBM (Tweedie)"
        reason = "Mật độ số 0 rất cao (WAPE/MASE bắt buộc; MAPE nổ inf%)"
    elif abs(float(acf_vals[52])) < 0.10 and sw_p > 0.05:
        rec = "Harmonic ARIMA / LightGBM (F1)"
        reason = "Chuỗi mượt gần chuẩn, không có mùa vụ năm ACF52~0"
    elif abs(float(acf_vals[52])) < 0.15:
        rec = "Harmonic ARIMA / Hybrid / Ridge"
        reason = "Có trôi mức / tự tương quan ngắn, mùa vụ năm yếu"
    else:
        rec = "LightGBM + ERA5 + Hybrid"
        reason = "Tự tương quan & mùa vụ năm mạnh (ACF52 >= 0.30), nhạy thời tiết"

    diag_rows.append({
        "atc_code": atc,
        "daily_mean": round(float(np.mean(y_d)), 2),
        "daily_zero_pct": round(float(100.0 * np.mean(y_d == 0)), 1),
        "adi": round(float(adi), 2),
        "cv2": round(float(cv2), 2),
        "sb_class": sb_class,
        "weekly_mean": round(float(np.mean(y_w)), 2),
        "weekly_cov": round(float(np.std(y_w, ddof=1) / np.mean(y_w)), 2),
        "weekly_zero_cnt": int(np.sum(y_w == 0)),
        "acf1_w": round(float(acf_vals[1]), 2),
        "acf52_w": round(float(acf_vals[52]), 2),
        "adf_p": round(adf_p, 4),
        "kpss_p": round(kpss_p, 4),
        "lb12_p": round(lb_p, 4),
        "mk_p": round(float(mk_p), 4),
        "sw_p": round(sw_p, 4),
        "model_recommendation": rec,
        "selection_rationale": reason,
    })

df_diag = pd.DataFrame(diag_rows)
df_diag.to_csv(ART_DIR / "eda_diagnostics_8atc.csv", index=False)
display(df_diag)

In [ ]:
# Biểu đồ 1: Lịch sử 6 năm theo tuần của 8 nhóm ATC
fig, axes = plt.subplots(4, 2, figsize=(13, 10), sharex=True)
atcs = sorted(df_weekly["atc_code"].unique())
for ax, atc in zip(axes.flat, atcs):
    g = df_weekly[df_weekly["atc_code"] == atc].sort_values("local_date")
    ax.plot(g["local_date"], g["qty"], lw=1.2, color="#0B5394")
    ax.set_title(f"{atc} (mean={g['qty'].mean():.1f}/W, zeros={(g['qty']==0).sum()}W)", fontsize=10, fontweight="bold")
    ax.grid(alpha=0.3)
fig.suptitle("Weekly Observed Sales Across 8 ATC Groups (2014-2019, N=302 Weeks)", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(ART_DIR / "eda_01_small_multiples_6y.png", dpi=300, bbox_inches="tight")
plt.show()

# Biểu đồ 2: Mặt phẳng phân loại Syntetos-Boylan (ADI vs CV^2)
fig, ax = plt.subplots(figsize=(8.4, 5.6))
colors = {"Smooth": "#137333", "Erratic": "#E37400", "Intermittent": "#C5221F", "Lumpy": "#8430CE"}
offsets = {
    "N02BE": (18, -14),
    "M01AB": (26, -2),
    "M01AE": (18, 12),
    "N02BA": (16, 4),
    "N05B": (14, 10),
    "R06": (14, 4),
    "R03": (14, 2),
    "N05C": (-120, 12),
}
for _, r in df_diag.iterrows():
    atc = r["atc_code"]
    ax.scatter(r["adi"], r["cv2"], s=130, color=colors[r["sb_class"]], edgecolor="black", zorder=3)
    dx, dy = offsets.get(atc, (12, 4))
    ax.annotate(
        f"{atc} ({r['sb_class']})",
        xy=(r["adi"], r["cv2"]),
        xytext=(dx, dy),
        textcoords="offset points",
        fontsize=9,
        fontweight="bold",
        arrowprops=dict(arrowstyle="-", color="#5F6368", lw=0.8),
    )
ax.axvline(1.32, color="gray", ls="--", lw=1.1)
ax.axhline(0.49, color="gray", ls="--", lw=1.1)
ax.set_xlim(0.85, 3.35)
ax.set_ylim(0.18, 0.90)
ax.set_xlabel("ADI (Average Demand Interval, Days)")
ax.set_ylabel("CV² (Squared Coefficient of Variation of Non-Zero Days)")
ax.set_title("Syntetos-Boylan Classification of 8 ATC Categories (Store 5631)", fontweight="bold")
fig.tight_layout()
fig.savefig(ART_DIR / "eda_02_syntetos_boylan_quadrant.png", dpi=300, bbox_inches="tight")
plt.show()

# Biểu đồ 3: STL Seasonality comparison (N02BE vs N05B)
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
for col_idx, atc in enumerate(["N02BE", "N05B"]):
    s = df_weekly[df_weekly["atc_code"] == atc].set_index("local_date")["qty"].sort_index()
    stl = STL(s, period=52, robust=True).fit()
    axes[0, col_idx].plot(stl.trend, color="#0B5394", lw=1.8)
    axes[0, col_idx].set_title(f"{atc} — STL Trend", fontweight="bold")
    axes[1, col_idx].plot(stl.seasonal, color="#1A73E8", lw=1.3)
    axes[1, col_idx].set_title(f"{atc} — STL Annual Seasonal (period=52)", fontweight="bold")
fig.tight_layout()
fig.savefig(ART_DIR / "eda_03_stl_n02be_vs_n05b.png", dpi=300, bbox_inches="tight")
plt.show()

# Biểu đồ 4: Weekly Cross-Correlation Preview (ERA5 Weather vs ATCs, lag 0..1W)
w_wk = df_weather.set_index("local_date")[["temp_mean", "temp_min", "humidity_mean", "daylight_duration_s"]].resample("W-SUN").mean()
merged_eda = df_weekly.merge(w_wk.reset_index(), on="local_date", how="left")
corr_rows = []
for atc, g in merged_eda.groupby("atc_code"):
    g = g.sort_values("local_date")
    row = {"atc_code": atc}
    for col in ["temp_mean", "temp_min", "humidity_mean", "daylight_duration_s"]:
        row[f"{col}_lag0"] = g["qty"].corr(g[col], method="spearman")
        row[f"{col}_lag1"] = g["qty"].corr(g[col].shift(1), method="spearman")
    corr_rows.append(row)
df_corr = pd.DataFrame(corr_rows).set_index("atc_code")
fig, ax = plt.subplots(figsize=(9.5, 4.8))
sns.heatmap(df_corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Spearman Correlation: ERA5 Weather (lag 0 / lag 1W) vs Weekly ATC Sales", fontweight="bold")
fig.tight_layout()
fig.savefig(ART_DIR / "eda_04_weather_ccf.png", dpi=300, bbox_inches="tight")
plt.show()